# 2.3.2 Native Sparse Attention(NSA)

motivation: 
- reduce attention computation 减少attention计算量 
- avoid perfomance degration - native / trainable 避免性能下降 - 原生/可训练

## Design
- **compressed attention**: holding global information/context
- **selection attention**: holding local information/context
- **sliding window attention**: holding related information/context

## Example

![architecture](../../figs/2.3.2-NSA-architecture.png)

*This picture is the overall architecture of NSA. (From NSA paper, see **Reference 1**)*

We have now calculated context length of 32 tokens. So we have the corresponding 32 KV cache. 
**Target**: caculate the new and less KV cache to compute with new query.

First, divide the 32 into 4 blocks, the block is of $len_b=8$

- **compression**: For each block, compress it into $len=len_b / c, c \le len_b$.
- **selection**: Select $s$ block from all the blocks.
- **sliding**: Select the former $l$ KV in the whole sequence. (e.g select 24-32 KV cache)

Last, use **gate** to gather all computed KV. 
$$KV_{new} = g^{cmp} o^{cmp} + g^{sle} o^{sle} + g^{win} o^{win}$$

The parameter $g$ is calculated from the linear layer output with the input $x_{32}$, which can be learned during training.

## Attention Implementation

In [ ]:
import torch

batch_size = 32
t = 32 # input sequence length
dim = 64 # hidden dimension

X = torch.randn(batch_size, t, dim)

W_q = torch.randn(dim, dim)
W_k = torch.randn(dim, dim)
W_v = torch.randn(dim, dim)

Q = X @ W_q
K = X @ W_k
V = X @ W_v

block_size = 8
num_blocks = t // block_size

### Compressed Attention

[single head version]

In [ ]:
W_k_cmp = torch.randn(block_size, 1)
W_v_cmp = torch.randn(block_size, 1)

## Train

## Optimization

## references

1. (paper) Native Sparse Attention: Hardware-Aligned and Natively Trainable Sparse Attention, https://arxiv.org/pdf/2502.11089
2. (blog) 【手撕NSA】DeepSeek新作-原生稀疏注意力-超长文(附代码), https://zhuanlan.zhihu.com/p/24841366485